# Aralin 11 - Agent-to-Agent (A2A) Protocol


## Setup


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Ano ang A2A Protocol?

Ang **Agent-to-Agent (A2A) protocol** ay isang bukas na pamantayan na nagpapahintulot sa mga AI agent na magkomunika,
maghanap ng isa't isa, at makipagtulungan — kahit na sila ay ginawa gamit ang iba't ibang mga framework o naka-host
sa iba't ibang mga serbisyo.

Pangunahing mga konsepto:

- **Pagtuklas** – Ang mga agent ay naglalathala ng *Agent Card* na naglalarawan ng kanilang mga kakayahan, na nagpapadali
  para sa ibang mga agent (o mga tagapangasiwa) na mahanap ang tamang espesyalista para sa isang gawain.
- **Pagpapalitan ng Mensahe** – Nagpapalitan ang mga agent ng mga nakaistrukturang mensahe sa pamamagitan ng isang pangkaraniwang protocol, kaya ang
  kahilingan mula sa isang agent ay maiintindihan at matutupad ng isa pa kahit na magkakaiba ang kanilang mga panloob na
  implementasyon.
- **Lifecycle ng Gawain** – Inilalarawan ng A2A ang mga estado tulad ng *submitted*, *working*, *completed*, at
  *failed*, na nagbibigay sa tagapangasiwa ng buong pananaw kung paano umuunlad ang isang ipinagkatiwalaang gawain.

Sa araling ito ay isinusuheto natin ang pagsasama-sama sa estilo ng A2A sa pamamagitan ng pagdudugtong ng tatlong mga espesyalistang travel agent
sa isang workflow kung saan bawat agent ay nag-aambag ng kanyang kadalubhasaan at ipinapasa ang mga resulta sa susunod.


## Paglikha ng Mga Espesyalistang Ahente ng Paglalakbay


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Pagkakatuwang ng Maramihang Ahente sa Pamamagitan ng Daloy ng Gawain

Pinagdugtong-dugtong namin ang tatlong ahente sa isang sunod-sunod na daloy ng gawain na kahawig ng A2A message passing:

1. **CurrencyExchangeAgent** ay tumatanggap ng kahilingan ng gumagamit at nagbibigay ng gabay sa pera.
2. **ActivityPlannerAgent** ay tumatanggap ng pinayamang konteksto at nagdaragdag ng mga rekomendasyon sa gawain.
3. **TravelManagerAgent** ay pinagsasama ang dalawang input sa isang panghuling buod ng paglalakbay.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Pag-unawa sa A2A sa Produksyon

Sa isang production environment, binubuksan ng A2A protocol ang makapangyarihang cross-service na mga scenario:

| Kakayahan | Paglalarawan |
|---|---|
| **Cross-framework interop** | Maaaring mag-delegate ang isang ahente na ginawa gamit ang isang framework ng mga gawain sa isang ahenteng ginawa gamit ang anumang ibang A2A-compliant na framework, na nagpapahintulot ng tunay na cross-organization interoperability. |
| **Mga hangganan ng serbisyo** | Maaaring mangyari ang mga ahente sa magkakahiwalay na microservices, mga rehiyon ng cloud, o kahit sa iba't ibang mga organisasyon habang patuloy na nakikipagtulungan nang walang patid. |
| **Dynamic discovery** | Maaaring mag-query ang isang orchestrator ng Agent Card registry sa runtime upang mahanap ang pinaka-angkop na espesyalista para sa isang partikular na sub-task. |
| **Streaming at push notifications** | Sinusuportahan ng A2A ang Server-Sent Events (SSE) para sa real-time na pag-update ng progreso at push notifications para sa mga long-running na gawain. |

Ang workflow na ginawa namin sa itaas ay isang pinasimpleng, in-process na bersyon ng pattern na ito. Sa totoong
deployment, bawat ahente ay magpapakita ng isang HTTP endpoint, magpapublish ng Agent Card, at makikipag-ugnayan
gamit ang A2A JSON-RPC protocol.


## Buod

Sa araling ito natutunan mo:

1. **Ano ang A2A protocol** — isang bukas na pamantayan para sa pagtuklas, pagmemensahe,
   at pamamahala ng gawain mula ahente sa ahente.
2. **Paano gumawa ng mga espesyal na ahente** — isang Currency Exchange agent, isang Activity Planner agent,
   at isang Travel Manager orchestrator.
3. **Paano ikabit ang mga ahente sa isang workflow** — gamit ang `WorkflowBuilder` upang imodelo ang sunud-sunod
   na pagpapadala ng mensahe sa pagitan ng mga ahente.
4. **Paano gumagana ang A2A sa produksyon** — nagpapahintulot ng kolaborasyon sa pagitan ng iba't ibang framework at serbisyo
   gamit ang dynamic na pagtuklas at streaming updates.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
